In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import time
import math
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import random
import os

In [ ]:
from pathlib import Path

dataset_name = "youtube_static.csv"
data_file = Path.cwd().parents[1] / "data" / "filtered" / dataset_name

df = pd.read_csv(data_file, parse_dates=["DATE"])
print("Loaded:", data_file)

In [3]:
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

In [ ]:
df = df.rename(columns={
    'ue_ident': 'item_id',
    'DATE': 'timestamp',
    'mac_dl_brate': 'target'
})

target_df = df[['item_id', 'timestamp', 'target']]
uni_data = TimeSeriesDataFrame(target_df)
uni_data.head()

In [5]:
prediction_length = 96

train_size = int(len(df) * 0.8)
train_data = uni_data.iloc[:train_size]
test_data = uni_data.iloc[train_size:]

In [ ]:
finetune_predictor = TimeSeriesPredictor(prediction_length=prediction_length, eval_metric="MAE", freq='ms').fit(
    train_data,
    hyperparameters={
        "Chronos": [
            {"model_path": "bolt_small", "ag_args": {"name_suffix": "ZeroShot"}},
            {"model_path": "bolt_small", "fine_tune": True, "ag_args": {"name_suffix": "FineTuned"}},
        ]
    },
    enable_ensemble=False,
    #time_limit=600,
)

In [ ]:
def rolling_chronos_forecast_all(predictor, test_data, prediction_length=96, stride=1, measure_time=False):

    results = []
    times = []

    test_data = test_data.copy()
    if not isinstance(test_data.index, pd.MultiIndex):
        raise ValueError("Expected test_data with MultiIndex (item_id, timestamp).")

    for item_id, series in test_data.groupby(level="item_id"):
        series = series.reset_index()  

        for start in range(0, len(series) - prediction_length, stride):
            context = series.iloc[: start + prediction_length]

            t0 = time.time()
            forecast = predictor.predict(context.set_index(["item_id", "timestamp"]))
            t1 = time.time()

            if measure_time:
                times.append(t1 - t0)

            
            forecast_mean = forecast.loc[item_id]["mean"].to_numpy().flatten()

            origin_ts = series["timestamp"].iloc[start]  
            forecast_ts = pd.date_range(start=origin_ts + pd.Timedelta(milliseconds=1),
                                        periods=prediction_length,
                                        freq="ms")

            df_forecast = pd.DataFrame({
                "item_id": item_id,
                "timestamp": forecast_ts,
                "mean": forecast_mean
            })
            results.append(df_forecast)

    forecasts_df = pd.concat(results, ignore_index=True)
    return (forecasts_df, times) if measure_time else forecasts_df


In [ ]:
rolling_preds, times = rolling_chronos_forecast_all(
    finetune_predictor, test_data, prediction_length=96, stride=1, measure_time=True
)

print(f"Average inference time per forecast: {np.mean(times):.4f} seconds")
print(f"Std of inference time: {np.std(times):.4f} seconds")
print(f"Total forecasts: {len(times)}")


In [ ]:
actual = test_data.reset_index()[["item_id", "timestamp", "target"]]

aligned = rolling_preds.merge(
    actual, on=["item_id", "timestamp"], how="inner"
)

In [ ]:
scaler = MinMaxScaler()
scaler.fit(train_data.reset_index()[["target"]])  

aligned["mean_scaled"] = scaler.transform(aligned[["mean"]].to_numpy())
aligned["target_scaled"] = scaler.transform(aligned[["target"]].to_numpy())

rmse_scaled = np.sqrt(mean_squared_error(aligned["target_scaled"], aligned["mean_scaled"]))
mae_scaled = mean_absolute_error(aligned["target_scaled"], aligned["mean_scaled"])

print(f"Scaled RMSE: {rmse_scaled:.4f}")
print(f"Scaled MAE: {mae_scaled:.4f}")


In [13]:
def block_average(df, value_col, item_col="item_id", timestamp_col="timestamp", block_len=96):
    df = df.sort_values([item_col, timestamp_col]).reset_index(drop=True)
    results = []

    for item_id, group in df.groupby(item_col):
        y = group[value_col].to_numpy()
        ts = group[timestamp_col].to_numpy()

        num_blocks = len(y) // block_len
        if num_blocks == 0:
            continue

        y_trunc = y[:num_blocks*block_len].reshape(num_blocks, block_len)
        ts_trunc = ts[:num_blocks*block_len].reshape(num_blocks, block_len)

        results.append(pd.DataFrame({
            item_col: item_id,
            "block_id": np.arange(num_blocks),
            "value_avg": y_trunc.mean(axis=1),
            "timestamp_mid": ts_trunc[:, block_len // 2]  # middle timestamp
        }))

    return pd.concat(results, ignore_index=True)


In [14]:
pred_blocked = block_average(aligned, value_col="mean", block_len=prediction_length)
actual_blocked = block_average(aligned, value_col="target", block_len=prediction_length)


In [ ]:
plt.figure(figsize=(12,6))
plt.plot(actual_blocked['timestamp_mid'], actual_blocked['value_avg'], label='Actual')
plt.plot(pred_blocked['timestamp_mid'], pred_blocked['value_avg'], label='Predicted')
plt.xlabel("Timestamp")
plt.ylabel("Downlink Bitrate")
plt.legend()
plt.show()


In [ ]:
results_dir = Path("../results/metrics/youtube_static")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([{
    "model": "Chronos-finetuning",
    "setting": "univariate",
    "dataset": "youtube_static",
    "rmse": rmse_scaled,
    "mae": mae_scaled,
}])

metrics_file = results_dir / "chronosft_uni_metrics.csv"
metrics_df.to_csv(metrics_file, index=False)

print("Saved metrics to:", metrics_file)